# Cars4You - Handout Project | Group 42
## Bayes Search Notebook
**Course:** MSc in Data Science and Advanced Analytics, NOVA IMS (2025/2026)  

**Author(s):** Group 42
- `Miguel Matos - 20221925`
- `Andre Nicolau - 20221918`
- `André Ferreira - 20250398`


---

#### <font> Table of Contents </font> <a class="anchor" id='toc'></a> 
1. [Imports](#Imports)  
2. [Modelling Testing](#modelling-testing) 
----

# Imports
[Back to TOC](#toc)

In [1]:
from utils.functions import *

First we have to clean the data of the test dataset

In [2]:
data = pd.read_csv("../data/test.csv", index_col= "carID")
pd.set_option("display.max_columns", None)

### "year"

In [3]:
data["year"] = np.clip(data["year"], 0, 2020)

In [4]:
data["year"] = data["year"].apply(lambda x: math.floor(x) if pd.notna(x) else x)
data["year"].describe()

count    31914.000000
mean      2017.080278
std          2.175345
min       1991.000000
25%       2016.000000
50%       2017.000000
75%       2019.000000
max       2020.000000
Name: year, dtype: float64

In [5]:
data["year"] = 2020 - data["year"]
data.rename(columns={"year": "car_age"}, inplace= True)
data

,Brand,model,car_age,transmission,mileage,fuelType,tax,mpg,engineSize,paintQuality%,previousOwners,hasDamage
carID,,,,,,,,,,,,
89856,Hyundai,I30,0.0,Automatic,30700.000000,petrol,205.0,41.5,1.6,61.0,3.0,0.0
106581,VW,Tiguan,3.0,Semi-Auto,-48190.655673,Petrol,150.0,38.2,2.0,60.0,2.0,0.0
80886,BMW,2 Series,4.0,Automatic,36792.000000,Petrol,125.0,51.4,1.5,94.0,2.0,0.0
100174,Opel,Grandland X,1.0,Manual,5533.000000,Petrol,145.0,44.1,1.2,77.0,1.0,0.0
81376,BMW,1 Series,1.0,Semi-Auto,9058.000000,Diesel,150.0,51.4,2.0,45.0,4.0,0.0
...,...,...,...,...,...,...,...,...,...,...,...,...
105775,VW,Tiguan,3.0,Manual,27575.000000,Petrol,145.0,46.3,1.4,94.0,1.0,0.0
81363,BMW,X2,0.0,Automatic,1980.000000,Petrol,145.0,34.0,2.0,39.0,3.0,0.0
76833,Audi,Q5,1.0,Semi-Auto,8297.000000,Diesel,145.0,38.2,2.0,88.0,4.0,0.0


### "mileage"

In [6]:
data.loc[data["mileage"] < 0, "mileage"] = np.nan

### "tax"

In [7]:
data.loc[data["tax"] < 0, "tax"] = np.nan

### "mpg"

In [8]:
data.loc[data["mpg"] < 0, "mpg"] = np.nan

### "engineSize"

In [9]:
data.loc[data["engineSize"] < 0, "engineSize"] = np.nan

In [10]:
data.loc[data["engineSize"] == 0, "engineSize"] = np.nan

### "paintQuality%"

In [11]:
data.drop("paintQuality%", axis = 1, inplace= True)

### "previousOwners"

In [12]:
data.loc[data["previousOwners"] < 0, "previousOwners"] = np.nan

In [13]:
data["previousOwners"] = data["previousOwners"].apply(lambda x: math.floor(x) if pd.notna(x) else x)
data["previousOwners"].describe()

count    31802.000000
mean         2.027703
std          1.438231
min          0.000000
25%          1.000000
50%          2.000000
75%          3.000000
max          6.000000
Name: previousOwners, dtype: float64

### "hasDamage"

In [14]:
data.drop("hasDamage", axis = 1, inplace= True)

#### Text normalization for categorical variables cleaning

In [15]:
data["model"] = data["model"].apply(normalize_text)
data["Brand"] = data["Brand"].apply(normalize_text)
data["transmission"] = data["transmission"].apply(normalize_text)
data["fuelType"] = data["fuelType"].apply(normalize_text)

### "brand" variable cleaning

In [16]:
correct_brands = {"Audi": ['aud', 'audi', 'udi', 'ud'],
                  "BMW": ['bmw', 'mw', 'bm'],
                  "Ford": ['ford', 'for', 'ord', 'or'],
                  "Hyundai" : ['hyundai', 'hyunda', 'yundai', 'yunda'],
                  "Mercedes" : ['mercedes','ercedes', 'mercede', 'ercede'],
                  "Skoda": ['skoda', 'koda', 'skod', 'kod'],
                  "Toyota": ['toyota', 'toyot', 'oyota'],
                  "Opel": ['opel', 'pel', 'ope', 'pe'],
                  "VW": ['vw', 'w','v']}

variant_to_brand = {variant: brand for brand, variants in correct_brands.items() for variant in variants}
data["Brand"] = data["Brand"].map(variant_to_brand)
data["Brand"].unique()

array(['Hyundai', 'VW', 'BMW', 'Opel', 'Ford', 'Mercedes', 'Skoda',
       'Toyota', 'Audi', nan], dtype=object)

### "model" variable cleaning

In [17]:
models_per_brand = pd.DataFrame(data.groupby(by =["Brand"])["model"].unique())
models_per_brand["model"] = models_per_brand["model"].apply(lambda x: [model for model in x if pd.notna(model)])
models_per_brand

,model
Brand,
Audi,"[tt, a4, a7, q2, a1, q5, a3, q3, q, a5, a6, a8..."
BMW,"[2 series, 1 series, x1, 5 series, 3 series, 4..."
Ford,"[fiesta, focus, ecosport, kuga, grand c-max, k..."
Hyundai,"[i30, ix20, i10, tucson, i40, ix35, i20, santa..."
Mercedes,"[b class, c class, m clas, e class, a class, x..."
Opel,"[grandland x, adam, zafira, corsa, insignia, m..."
Skoda,"[superb, fabia, octavia, karoq, kamiq, kodiaq,..."
Toyota,"[aygo, land cruiser, yaris, rav4, auris, corol..."
VW,"[tiguan, up, golf, passat, caravelle, polo, sh..."


In [18]:
models_per_brand["model"] = models_per_brand["model"].apply(lambda x: [re.sub(r'\bclas\b', 'class', str(model)) for model in x])
models_per_brand

,model
Brand,
Audi,"[tt, a4, a7, q2, a1, q5, a3, q3, q, a5, a6, a8..."
BMW,"[2 series, 1 series, x1, 5 series, 3 series, 4..."
Ford,"[fiesta, focus, ecosport, kuga, grand c-max, k..."
Hyundai,"[i30, ix20, i10, tucson, i40, ix35, i20, santa..."
Mercedes,"[b class, c class, m class, e class, a class, ..."
Opel,"[grandland x, adam, zafira, corsa, insignia, m..."
Skoda,"[superb, fabia, octavia, karoq, kamiq, kodiaq,..."
Toyota,"[aygo, land cruiser, yaris, rav4, auris, corol..."
VW,"[tiguan, up, golf, passat, caravelle, polo, sh..."


In [19]:
brands = models_per_brand.index.tolist()

for models, brand in zip(models_per_brand["model"], brands):
    canonical_names = {}  # to store canonical form for each variant

    for model in models:
        if model in canonical_names:
            continue  # already assigned

        # adaptive threshold
        threshold = 90 if len(model) < 4 else 75

        # find similar models within this brand
        matches = process.extract(model, models, scorer=fuzz.token_sort_ratio)
        close = [m for m, score, _ in matches if score >= threshold]
        for c in close:
            canonical_names[c] = model  # group under canonical name

    data.loc[data["Brand"] == brand , "model"] = data.loc[data["Brand"] == brand, "model"].map(canonical_names).fillna(data.loc[data["Brand"] == brand, "model"])

data["model"] = data["model"].str.capitalize()

### "fuelType" variable cleaning

In [20]:
correct_fueltype = {"Diesel": ["diesel", "iesel", "diese", "iese"],
                    "Petrol": ["petrol", "petro", "etrol", "etro"],
                    "Hybrid": ["hybrid", "ybrid", "hybri", "ybri"],
                    "Other": ["other", "ther", "othe"],
                    "Eletric": "eletric"}

variant_to_fuelType = {variant: fuelType for fuelType, variants in correct_fueltype.items() for variant in variants}
data["fuelType"] = data["fuelType"].map(variant_to_fuelType)
data["fuelType"].unique()

array(['Petrol', 'Diesel', 'Hybrid', nan, 'Other'], dtype=object)

### "transmission" variable cleaning

In [21]:
correct_transmissions = {
    "Manual": ["manual", "anual", "manua", "anua"],
    "Semi-Auto": ["semi-auto", "emi-auto", "semi-aut", "emi-aut"],
    "Automatic": ["automatic", "automati", "utomatic", "utomati"],
    "Unknown": ["unknown", "unknow", "nknown", "nknow"],"Other": ["other"]}

variant_to_transmissions = {variant: transmissions for transmissions, variants in correct_transmissions.items() for variant in variants}
data["transmission"] = data["transmission"].map(variant_to_transmissions)
data["transmission"].unique()

array(['Automatic', 'Semi-Auto', 'Manual', 'Unknown', nan, 'Other'],
      dtype=object)

### "Brand" missing values filling

In [22]:
brand_per_model = (
    data.groupby("model")["Brand"]
        .unique()                       # get unique brands per model (still lists)
        .apply(lambda x: [b for b in x if pd.notna(b)])  # remove NaNs
        .apply(lambda x: x[0] if len(x) > 0 else None) # take first brand as string
        .to_dict()                        # convert to dictionary
)

brand_per_model

{'1 series': None,
 '180': 'Mercedes',
 '2 series': None,
 '3 serie': 'BMW',
 '3 series': None,
 '4 series': 'BMW',
 '5 serie': 'BMW',
 '5 series': None,
 '6 series': 'BMW',
 '7 series': 'BMW',
 '8 series': 'BMW',
 'A': 'Audi',
 'A clas': 'Mercedes',
 'A class': None,
 'A1': 'Audi',
 'A3': 'Audi',
 'A4': 'Audi',
 'A5': 'Audi',
 'A6': 'Audi',
 'A7': 'Audi',
 'A8': 'Audi',
 'Adam': 'Opel',
 'Agila': 'Opel',
 'Amarok': 'VW',
 'Amica': None,
 'Ampera': 'Opel',
 'Antara': 'Opel',
 'Arteon': 'VW',
 'Astra': 'Opel',
 'Auri': 'Toyota',
 'Auris': None,
 'Avensis': 'Toyota',
 'Aygo': 'Toyota',
 'B clas': 'Mercedes',
 'B class': 'Mercedes',
 'B-ma': 'Ford',
 'B-max': None,
 'Beetle': 'VW',
 'C clas': 'Mercedes',
 'C class': None,
 'C-hr': 'Toyota',
 'C-max': 'Ford',
 'Caddy': 'VW',
 'Caddy maxi life': 'VW',
 'California': 'VW',
 'Camry': 'Toyota',
 'Caravelle': 'VW',
 'Cascada': 'Opel',
 'Cc': 'VW',
 'Citigo': 'Skoda',
 'Cl clas': 'Mercedes',
 'Cl class': 'Mercedes',
 'Cla clas': 'Mercedes',
 'Cl

In [23]:
data["Brand"] = data["Brand"].fillna(data["model"].map(brand_per_model))

In [24]:
test_cleaned = data.copy()

In [25]:
test_cleaned

,Brand,model,car_age,transmission,mileage,fuelType,tax,mpg,engineSize,previousOwners
carID,,,,,,,,,,
89856,Hyundai,I30,0.0,Automatic,30700.0,Petrol,205.0,41.5,1.6,3.0
106581,VW,Tiguan,3.0,Semi-Auto,NaN,Petrol,150.0,38.2,2.0,2.0
80886,BMW,6 series,4.0,Automatic,36792.0,Petrol,125.0,51.4,1.5,2.0
100174,Opel,Grandland x,1.0,Manual,5533.0,Petrol,145.0,44.1,1.2,1.0
81376,BMW,6 series,1.0,Semi-Auto,9058.0,Diesel,150.0,51.4,2.0,4.0
...,...,...,...,...,...,...,...,...,...,...
105775,VW,Tiguan,3.0,Manual,27575.0,Petrol,145.0,46.3,1.4,1.0
81363,BMW,X2,0.0,Automatic,1980.0,Petrol,145.0,34.0,2.0,3.0
76833,Audi,Q5,1.0,Semi-Auto,8297.0,Diesel,145.0,38.2,2.0,4.0


In [27]:
test_cleaned.to_csv("../data/Test_clean.csv")